
# Automatic BB

### Image comparing

In [1]:
import cv2

img_path1 = "editing/samples/sk1.png"
img_path2 = "editing/samples/sk2.png"
gen_img1 = "editing/samples/gen_img1.png"
gen_img2 = "editing/samples/gen_img2.png"

%load_ext autoreload
%autoreload 2

In [11]:
img1 = cv2.imread(img_path1, cv2.IMREAD_COLOR)
img2 = cv2.imread(img_path2, cv2.IMREAD_COLOR)

diff = cv2.absdiff(img1, img2)  # per-pixel absolute difference
cv2.imwrite("editing/samples/diff1.png", diff)

True

In [14]:
import cv2

# img_path1 = "editing/samples/image_1.png"
# img_path2 = "editing/samples/image_2.png"

img1 = cv2.imread(img_path1, cv2.IMREAD_COLOR)
img2 = cv2.imread(img_path2, cv2.IMREAD_COLOR)

# 1. Get the absolute per-pixel difference
diff = cv2.absdiff(img1, img2)

# 2. Convert the difference to grayscale
# This merges the BGR channels into a single intensity map so we can threshold it cleanly.
gray_diff = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)

# 3. Define your threshold value (0 to 255)
# A value of 30 is a great starting point to ignore tiny sensor noise or jpeg compression.
# Increase this number if it's picking up too much background noise.
threshold_value = 200 

# 4. Apply the Binary Threshold
# If the difference is > 30, force it to 255 (Pure White). 
# If it's <= 30, force it to 0 (Pure Black).
_, thresh_mask = cv2.threshold(gray_diff, threshold_value, 255, cv2.THRESH_BINARY)

# Save both so you can compare the raw difference vs the clean threshold mask
cv2.imwrite("editing/samples/img_diff1_raw.png", diff)
cv2.imwrite("editing/samples/diff1.png", thresh_mask)

True

In [2]:
import torch
torch.device("cuda" if torch.cuda.is_available() else "cpu")

device(type='cuda')

In [2]:
from depth_anything_3.api import DepthAnything3
import cv2
import torch
import time
import numpy as np

class DA3:
    def __init__(self):
        # model = DepthAnything3.from_pretrained("depth-anything/DA3MONO-LARGE")
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        model = DepthAnything3.from_pretrained("depth-anything/DA3NESTED-GIANT-LARGE-1.1")
        # model = DepthAnything3.from_pretrained("depth-anything/DA3METRIC-LARGE")
        model = model.to(device)
        model.eval()
        print(f"Model loaded on {device}")
        
        self.run_times = []
        self.model = model
        self.device = device

    def forward(self, img_path):
        start = time.perf_counter()

        original_image = cv2.imread(img_path)
        H_orig, W_orig = original_image.shape[:2]
        longest_edge = max(H_orig, W_orig)
        optimal_res = int(round(longest_edge / (14.0)) * 14)
        process_res = min(optimal_res, 1330)
        print(f"Running native inference at process_res: {process_res}")

        prediction = self.model.inference(image=[img_path], process_res=process_res)

        depth = prediction.depth[0] # Depth in [m].
        print("depth:", depth.shape)

        depth_resized = cv2.resize(
            depth, 
            (W_orig, H_orig), 
            interpolation=cv2.INTER_LINEAR  # Change to cv2.INTER_NEAREST if edges stretch in 3D
        )
        depth_resized = depth_resized.astype(np.float32)
        print("depth_resized:", depth_resized.shape)

        if prediction.intrinsics is None:
            return depth_resized, None, None, None, None, None, 

        H_pred, W_pred = depth.shape

        scale_x = W_orig / W_pred
        scale_y = H_orig / H_pred
        fx = prediction.intrinsics[0, 0, 0] * scale_x
        fy = prediction.intrinsics[0, 1, 1] * scale_y
        cx = prediction.intrinsics[0, 0, 2] * scale_x
        cy = prediction.intrinsics[0, 1, 2] * scale_y

        h, w = depth_resized.shape
        SENSOR_HEIGHT_MM = 24.0  # Standard full-frame sensor height
        focal_length = (fy / h) * SENSOR_HEIGHT_MM
        print("focal_length:", focal_length)

        end = time.perf_counter()
        runtime = round(end - start, 3)
        self.run_times.append(runtime)

        return depth_resized, focal_length, fx, fy, cx, cy

    def mean_runtime (self):
        arr = np.array(self.run_times)
        return arr.mean()

[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70


In [3]:
da3 = DA3()

WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.4.0+cu121 with CUDA 1201 (you have 2.5.1+cu121)
    Python  3.10.11 (you have 3.10.20)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


[INFO ] using SwiGLU layer as FFN
[INFO ] using MLP layer as FFN
Model loaded on cuda


In [4]:
depth, focal_length, fx, fy, cx, cy = da3.forward(gen_img2)

Running native inference at process_res: 294
[INFO ] Processed Images Done taking 0.05037951469421387 seconds. Shape:  torch.Size([1, 3, 294, 294])
[INFO ] Model Forward Pass Done. Time: 1.1506052017211914 seconds
[INFO ] Conversion to Prediction Done. Time: 0.005682706832885742 seconds
depth: (294, 294)
depth_resized: (288, 288)
focal_length: 72.07678


In [13]:


from editing.reconstruction import extract_changes

original_image = cv2.imread(gen_img2)
rgb_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)

extract_changes(img_path1, img_path2, gen_img2, depth, fx, fy, cx, cy)

Pipeline complete. Files saved to 'editing/output'.


{'min': [-0.04654489120395628, -0.19095811023485357, -2.4213485717773438],
 'max': [0.18804063835213286, 0.025787993806944274, -2.1105422973632812]}

In [ ]:
original_image = cv2.imread(gen_img2)
diff_image = cv2.imread("editing/samples/diff1.png")

# Convert the BGR image to an RGB numpy array
rgb_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
diff_image = cv2.cvtColor(diff_image, cv2.COLOR_BGR2RGB)
diff_image.mean()

np.float64(4.998)

In [30]:
%load_ext autoreload
%autoreload 2

from editing.reconstruction import save_pointcloud_diff, get_3dbb

save_pointcloud_diff(rgb_image, depth, diff_image, fx, fy, cx, cy, depth_max=30)
get_3dbb(depth, diff_image, fx, fy, cx, cy, depth_max=30)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Computed Strict 3DBB:
X: 0.011 -> 0.209
Y: 0.000 -> 0.129
Z: 2.118 -> 2.948 (Margin: 0.05)
Successfully isolated the dense volume! Saved 173 points to cloud_dense_S.ply


{'min': [0.5149478771524163, 0.7658787978224001, 0.08926086127758026],
 'max': [0.6112390556157304, 0.8700712998963477, 0.3095947802066803]}